# Dataset 

In [ ]:
import os
DATA_ROOT = os.environ.get('PANSORI_DATA_ROOT', '../../Pansori_Data')
import os
import math
import json
import random
import unicodedata
from pathlib import Path
from copy import deepcopy
from abc import abstractmethod
from collections import Counter
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torchaudio
import soundfile as sf
from torch.utils.data import Dataset
from torchaudio.transforms import TimeStretch, FrequencyMasking
from torchaudio.transforms import Spectrogram, MelScale, AmplitudeToDB
from typing import List, Optional

class BaseDataset(Dataset):
    def __init__(self, data_dir: str,
                 song_list: Optional[List[str]] = None,
                 sr: int = 16000,
                 n_fft: int = 1024,
                 hop_length: int = 512,
                 n_mels: int = 128,
                 segment_duration: float = 30.0,
                 min_duration: float = 20.0,
                 shift_cents_range: tuple = (-10, 10),
                 mask_duration: float = 5.0,
                 aug_prob: float = 0.7,
                 is_train: bool = False):

        self.data_dir = data_dir
        self.song_list = song_list
        self.sr = sr
        self.n_fft: int = 1024
        self.hop_length: int = 512
        self.n_mels: int = 128
        self.segment_duration = segment_duration
        self.min_duration = min_duration
        self.is_train = is_train
        self.shift_cents_range = shift_cents_range
        self.mask_duration = mask_duration
        self.aug_prob = aug_prob

        self.segment_frames = int(segment_duration * sr / hop_length)
        self.min_frames = int(min_duration * sr / hop_length)

        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels
        )

        self.db_transform = torchaudio.transforms.AmplitudeToDB()

        self.loaded_data, self.song_meta = self.load_data()
        self.song_names = list(self.loaded_data.keys())

    def get_mode_label(self, filename: str) -> int:
        return 0 if "우조" in filename else 1

    def load_audio(self, audio_file):
        # Use soundfile directly to avoid torchcodec dependency
        audio, file_sr = sf.read(audio_file, dtype='float32')
        # Convert to torch tensor and add channel dimension
        audio = torch.from_numpy(audio).T  # Transpose to [channels, samples]
        if audio.ndim == 1:
            audio = audio.unsqueeze(0)  # Add channel dimension if mono

        # Resample if needed
        if file_sr != self.sr:
            audio = torchaudio.functional.resample(audio, orig_freq=file_sr, new_freq=self.sr)

        # Convert to mono if stereo
        if audio.shape[0] > 1:
            audio = audio.mean(dim=0, keepdim=True)

        return audio

    def apply_audio_augmentation(self, audio: torch.Tensor) -> torch.Tensor:
        if not self.is_train or random.random() > self.aug_prob:
            return audio

        # Noise
        if random.random() < 0.3:
            noise_level = random.uniform(0.001, 0.005)
            noise = torch.randn_like(audio) * noise_level
            audio = audio + noise

        # Gain
        if random.random() < 0.3:
            gain = random.uniform(0.7, 1.3)
            audio = audio * gain

        # Pitch Shift here
        return audio

    def compute_mel(self, audio: torch.Tensor) -> torch.Tensor:
        mel_spec = self.mel_transform(audio)
        mel_spec = self.db_transform(mel_spec).squeeze(0) # remove channel dim
        mel_spec = mel_spec / 100.0 # Normalize
        return mel_spec

    def pad_audio(self, audio: torch.Tensor, target_duration: float) -> torch.Tensor:
        target_samples = int(target_duration * self.sr)
        current_samples = audio.shape[1]

        if current_samples < target_samples:
            pad_samples = target_samples - current_samples
            audio = torch.nn.functional.pad(audio, (0, pad_samples))

        return audio

    def load_data(self):
        loaded_data = {}
        song_meta = {}

        if self.song_list:
            files_to_load = [f"{song}.wav" if not song.endswith('.wav') else song
                             for song in self.song_list]
        else:
            files_to_load = [f for f in os.listdir(self.data_dir)
                           if f.endswith('.wav')]

        ujo_count = 0
        gmjo_count = 0

        for filename in files_to_load:
            if "우조" not in filename and "계면조" not in filename:
                continue

            audio_path = os.path.join(self.data_dir, filename)
            audio = self.load_audio(audio_path)
            audio_duration = audio.shape[1] / self.sr
            if audio_duration < self.min_duration:
                print(f"Skipping {filename}: duration {audio_duration:.2f}s < {self.min_duration}s")
                continue

            if audio_duration < self.segment_duration:
                audio = self.pad_audio(audio, self.segment_duration)
                audio_duration = self.segment_duration

            mel_spec = self.compute_mel(audio)

            basename = filename.replace('.wav', '')
            loaded_data[basename] = {'mel': mel_spec}
            song_meta[basename] = {
                'total_frames': mel_spec.shape[1],
                'total_duration': audio_duration,
                'mode_label': self.get_mode_label(basename)
            }

            if "우조" in basename:
                ujo_count += 1
            else:
                gmjo_count += 1

        print(f"Loaded {len(loaded_data)} songs")
        print(f"우조: {ujo_count}, 계면조: {gmjo_count}")
        return loaded_data, song_meta

    def random_crop(self, mel_spec: torch.Tensor) -> torch.Tensor:
        total_frames = mel_spec.shape[1]

        if total_frames <= self.segment_frames:
            return mel_spec

        start_frame = random.randint(0, total_frames - self.segment_frames)

        end_frame = start_frame + self.segment_frames

        return mel_spec[:, start_frame:end_frame]

    def apply_spec_augmentation(self, mel_spec: torch.Tensor) -> torch.Tensor:

        if random.random() < 0.5:
            mask_frames = int(self.mask_duration * self.sr / self.hop_length)
            start_frame = random.randint(0, max(0, mel_spec.shape[1] - mask_frames))
            mel_spec[:, start_frame:start_frame + mask_frames] = 0

        if random.random() < 0.5:
            mask_bins = random.randint(5, 15)
            start_bin = random.randint(0, max(0, mel_spec.shape[0] - mask_bins))
            mel_spec[start_bin: start_bin + mask_bins, :] = 0

        return mel_spec

    def __len__(self):
        return len(self.song_names)

    def __getitem__(self, idx):
        song_name = self.song_names[idx]

        mel_spec = self.loaded_data[song_name]['mel']
        mode_label = self.song_meta[song_name]['mode_label']

        mel_segment = self.random_crop(mel_spec)

        if self.is_train and random.random() < self.aug_prob:
            mel_segment = self.apply_spec_augmentation(mel_segment)

        return {
            'mel': mel_segment,
            'mode_label': mode_label,
            'song_name': song_name
        }

## Dataset Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader

# Example usage and visualization
def visualize_dataset_samples(dataset, num_samples=4, save_path=None):
    """Visualize random samples from the dataset"""
    fig, axes = plt.subplots(num_samples, 2, figsize=(15, 4 * num_samples))

    if num_samples == 1:
        axes = axes.reshape(1, -1)

    for i in range(num_samples):
        # Get random sample
        idx = np.random.randint(0, len(dataset))
        sample = dataset[idx]

        mel_spec = sample['mel'].numpy()  # [n_mels, time_frames]
        mode_label = sample['mode_label']
        song_name = sample['song_name']

        mode_name = "우조" if mode_label == 0 else "계면조"

        # Plot full mel spectrogram
        im1 = axes[i, 0].imshow(mel_spec, aspect='auto', origin='lower', cmap='viridis')
        axes[i, 0].set_title(f'{song_name}\nMode: {mode_name}', fontsize=10)
        axes[i, 0].set_xlabel('Time Frames')
        axes[i, 0].set_ylabel('Mel Bins')
        plt.colorbar(im1, ax=axes[i, 0])

        # Plot zoomed-in portion (first 5 seconds)
        zoom_frames = int(5 * dataset.sr / dataset.hop_length)
        mel_zoom = mel_spec[:, :zoom_frames]
        im2 = axes[i, 1].imshow(mel_zoom, aspect='auto', origin='lower', cmap='viridis')
        axes[i, 1].set_title(f'First 5 seconds (zoomed)', fontsize=10)
        axes[i, 1].set_xlabel('Time Frames')
        axes[i, 1].set_ylabel('Mel Bins')
        plt.colorbar(im2, ax=axes[i, 1])

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved visualization to {save_path}")

    plt.show()


def visualize_augmentation_effect(dataset, song_idx=0, num_augmentations=4):
    """Compare original vs augmented spectrograms"""

    # Temporarily set to train mode to enable augmentation
    original_is_train = dataset.is_train
    dataset.is_train = True

    fig, axes = plt.subplots(2, num_augmentations, figsize=(20, 8))

    sample = dataset[song_idx]
    song_name = sample['song_name']
    mode_label = sample['mode_label']
    mode_name = "우조" if mode_label == 0 else "계면조"

    # Get original (no augmentation)
    dataset.is_train = False
    original_sample = dataset[song_idx]
    original_mel = original_sample['mel'].numpy()

    dataset.is_train = True

    for i in range(num_augmentations):
        # Get augmented version
        aug_sample = dataset[song_idx]
        aug_mel = aug_sample['mel'].numpy()

        # Plot original
        im1 = axes[0, i].imshow(original_mel, aspect='auto', origin='lower', cmap='viridis')
        if i == 0:
            axes[0, i].set_ylabel('Original\nMel Bins', fontsize=10)
        axes[0, i].set_xlabel('Time Frames')
        axes[0, i].set_title(f'Sample {i+1}')

        # Plot augmented
        im2 = axes[1, i].imshow(aug_mel, aspect='auto', origin='lower', cmap='viridis')
        if i == 0:
            axes[1, i].set_ylabel('Augmented\nMel Bins', fontsize=10)
        axes[1, i].set_xlabel('Time Frames')

    plt.suptitle(f'Augmentation Comparison: {song_name} ({mode_name})', fontsize=14)
    plt.tight_layout()
    plt.show()

    # Restore original state
    dataset.is_train = original_is_train


def visualize_random_crops(dataset, song_idx=0, num_crops=6):
    """Show different random crops from the same song"""

    song_name = dataset.song_names[song_idx]
    full_mel = dataset.loaded_data[song_name]['mel'].numpy()
    mode_label = dataset.song_meta[song_name]['mode_label']
    mode_name = "우조" if mode_label == 0 else "계면조"
    total_duration = dataset.song_meta[song_name]['total_duration']

    fig, axes = plt.subplots(num_crops + 1, 1, figsize=(15, 2 * (num_crops + 1)))

    # Plot full spectrogram
    im = axes[0].imshow(full_mel, aspect='auto', origin='lower', cmap='viridis')
    axes[0].set_title(f'Full song: {song_name} ({mode_name}) - {total_duration:.1f}s', fontsize=12)
    axes[0].set_ylabel('Mel Bins')
    plt.colorbar(im, ax=axes[0])

    # Plot random crops
    original_is_train = dataset.is_train
    dataset.is_train = True

    for i in range(num_crops):
        sample = dataset[song_idx]
        cropped_mel = sample['mel'].numpy()

        im = axes[i + 1].imshow(cropped_mel, aspect='auto', origin='lower', cmap='viridis')
        axes[i + 1].set_title(f'Random crop {i+1} (30s)', fontsize=10)
        axes[i + 1].set_ylabel('Mel Bins')
        if i == num_crops - 1:
            axes[i + 1].set_xlabel('Time Frames')

    plt.tight_layout()
    plt.show()

    dataset.is_train = original_is_train


def visualize_batch(dataloader, num_batches=2):
    """Visualize batches from dataloader"""

    for batch_idx, batch in enumerate(dataloader):
        if batch_idx >= num_batches:
            break

        mel_batch = batch['mel']  # [batch_size, n_mels, time_frames]
        mode_labels = batch['mode_label']
        song_names = batch['song_name']

        batch_size = mel_batch.shape[0]

        fig, axes = plt.subplots(1, batch_size, figsize=(5 * batch_size, 4))

        if batch_size == 1:
            axes = [axes]

        for i in range(batch_size):
            mel = mel_batch[i].numpy()  # [n_mels, time_frames]
            mode_label = mode_labels[i].item()
            song_name = song_names[i]
            mode_name = "우조" if mode_label == 0 else "계면조"

            im = axes[i].imshow(mel, aspect='auto', origin='lower', cmap='viridis')
            axes[i].set_title(f'{song_name}\n{mode_name}', fontsize=10)
            axes[i].set_xlabel('Time Frames')
            if i == 0:
                axes[i].set_ylabel('Mel Bins')
            plt.colorbar(im, ax=axes[i])

        plt.suptitle(f'Batch {batch_idx + 1}', fontsize=14)
        plt.tight_layout()
        plt.show()


def print_dataset_statistics(dataset):
    """Print detailed statistics about the dataset"""

    print("\n" + "="*60)
    print("DATASET STATISTICS")
    print("="*60)

    print(f"\nTotal songs: {len(dataset)}")

    # Mode distribution
    mode_counts = {0: 0, 1: 0}
    durations = []
    frame_counts = []

    for song_name in dataset.song_names:
        mode_label = dataset.song_meta[song_name]['mode_label']
        duration = dataset.song_meta[song_name]['total_duration']
        frames = dataset.song_meta[song_name]['total_frames']

        mode_counts[mode_label] += 1
        durations.append(duration)
        frame_counts.append(frames)

    print(f"\nMode distribution:")
    print(f"  우조 (0): {mode_counts[0]} songs ({mode_counts[0]/len(dataset)*100:.1f}%)")
    print(f"  계면조 (1): {mode_counts[1]} songs ({mode_counts[1]/len(dataset)*100:.1f}%)")

    print(f"\nDuration statistics:")
    print(f"  Mean: {np.mean(durations):.2f}s")
    print(f"  Median: {np.median(durations):.2f}s")
    print(f"  Min: {np.min(durations):.2f}s")
    print(f"  Max: {np.max(durations):.2f}s")
    print(f"  Std: {np.std(durations):.2f}s")

    print(f"\nFrame count statistics:")
    print(f"  Mean: {np.mean(frame_counts):.1f} frames")
    print(f"  Median: {np.median(frame_counts):.1f} frames")
    print(f"  Min: {np.min(frame_counts)} frames")
    print(f"  Max: {np.max(frame_counts)} frames")

    print(f"\nSegment settings:")
    print(f"  Segment duration: {dataset.segment_duration}s")
    print(f"  Segment frames: {dataset.segment_frames}")
    print(f"  Sample rate: {dataset.sr} Hz")
    print(f"  Hop length: {dataset.hop_length}")

    # Memory estimate
    total_memory = 0
    for song_name in dataset.song_names:
        mel_shape = dataset.loaded_data[song_name]['mel'].shape
        # torch.float32 = 4 bytes
        memory_mb = (mel_shape[0] * mel_shape[1] * 4) / (1024 * 1024)
        total_memory += memory_mb

    print(f"\nMemory usage:")
    print(f"  Total: {total_memory:.2f} MB")
    print(f"  Per song average: {total_memory/len(dataset):.2f} MB")

    print("="*60 + "\n")


# Main execution example
if __name__ == "__main__":

    # Initialize dataset
    train_dataset = BaseDataset(
        data_dir="../data/audio",
        sr=16000,
        n_fft=1024,
        hop_length=512,
        n_mels=128,
        segment_duration=30.0,
        min_duration=20.0,
        aug_prob=0.7,
        is_train=True
    )

    val_dataset = BaseDataset(
        data_dir="../data/audio",
        sr=16000,
        n_fft=1024,
        hop_length=512,
        n_mels=128,
        segment_duration=30.0,
        min_duration=20.0,
        aug_prob=0.0,  # No augmentation for validation
        is_train=False
    )

    # Print dataset shape information
    print("\n" + "="*60)
    print("DATASET SHAPE INFORMATION")
    print("="*60)
    print(f"Number of samples in dataset: {len(train_dataset)}")

    # Get a sample to show its shape
    sample = train_dataset[0]
    print(f"\nSample shapes:")
    print(f"  Mel spectrogram shape: {sample['mel'].shape}")
    print(f"  Mode label: {sample['mode_label']} (type: {type(sample['mode_label'])})")
    print(f"  Song name: {sample['song_name']}")

    # Show batch shapes
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    batch = next(iter(train_loader))
    print(f"\nBatch shapes (batch_size=8):")
    print(f"  Mel batch: {batch['mel'].shape}")
    print(f"  Mode labels: {batch['mode_label'].shape}")
    print(f"  Song names: {len(batch['song_name'])} items")
    print("="*60 + "\n")

    # Print statistics
    print_dataset_statistics(train_dataset)

    # Visualize random samples
    print("Visualizing random samples...")
    visualize_dataset_samples(train_dataset, num_samples=4, save_path='dataset_samples.png')

    # Visualize augmentation effect
    print("\nVisualizing augmentation effects...")
    visualize_augmentation_effect(train_dataset, song_idx=0, num_augmentations=4)

    # Visualize random crops from same song
    print("\nVisualizing random crops from the same song...")
    visualize_random_crops(train_dataset, song_idx=0, num_crops=6)

    # Create dataloader
    train_loader = DataLoader(
        train_dataset,
        batch_size=8,
        shuffle=True,
        num_workers=4,
        pin_memory=True
    )

    # Visualize batches
    print("\nVisualizing batches from dataloader...")
    visualize_batch(train_loader, num_batches=2)

    # Test iteration speed
    print("\nTesting iteration speed...")
    import time

    start_time = time.time()
    for batch_idx, batch in enumerate(train_loader):
        if batch_idx >= 10:
            break
    elapsed = time.time() - start_time

    print(f"Time for 10 batches: {elapsed:.2f}s")
    print(f"Average time per batch: {elapsed/10:.3f}s")

    # Show sample shapes
    print("\nSample batch shapes:")
    sample_batch = next(iter(train_loader))
    print(f"  mel: {sample_batch['mel'].shape}")
    print(f"  mode_label: {sample_batch['mode_label'].shape}")
    print(f"  song_names: {len(sample_batch['song_name'])} items")

# Model 

In [ ]:
import torch.nn as nn

class Conv2DBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, padding, dilation):
        super().__init__()
        self.conv_norm = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=padding, dilation=dilation),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv_norm(x)

class SegConv2DGRU(nn.Module):
    def __init__(self,
                 n_mels=128,
                 n_frames=938,
                 num_conv_layers=3,
                 conv_channels=128,
                 kernel_size=3,
                 dilation=(1,2),
                 pool_size=(2,1),
                 dropout=0.3,
                 gru_hidden_dim=128,
                 num_gru_layers=3,
                 num_classes=2,
                 use_gradual_channels=False):

        super().__init__()

        self.n_mels = n_mels
        self.n_frames = n_frames
        self.num_conv_layers = num_conv_layers
        self.conv_channels = conv_channels
        self.kernel_size = kernel_size
        self.dilation = dilation
        self.pool_size = pool_size
        self.dropout = dropout
        self.gru_hidden_dim = gru_hidden_dim
        self.num_gru_layers = num_gru_layers
        self.num_classes = num_classes

        # Calculate Conv Layer Parameters
        self.conv_param = self.calc_conv_params(use_gradual_channels)

        # Build CNN Encoder
        self.encoder = self.build_encoder(kernel_size, dilation)

        # Calculate dimensions after CNN
        out_freq = self.calc_output_freq()
        out_time = self.calc_output_time()

        gru_input_size = self.conv_param[-1]['output_channel'] * out_freq

        self.gru = nn.GRU(
            input_size=gru_input_size,
            hidden_size=gru_hidden_dim,
            num_layers=num_gru_layers,
            batch_first=True,
            dropout=self.dropout,
            bidirectional=True
        )

        self.fc = nn.Linear(gru_hidden_dim * 2, num_classes)

    def calc_conv_params(self, use_gradual_channels):
        '''Calculate input and outptu channels for each conv layer'''
        params = []

        if use_gradual_channels:
            # Gradually increase channels: 64 -> 128 -> 128
            for i in range(self.num_conv_layers):
                if i == 0:
                    in_ch = 1
                    out_ch = self.conv_channels // 2
                elif i == self.num_conv_layers - 1:
                    in_ch = self.conv_channels // 2
                    out_ch = self.conv_channels
                else:
                    scale_factor = 1 / (2 ** (self.num_conv_layers - 1 - i))
                    in_ch = int(self.conv_channels * (scale_factor / 2))
                    out_ch = int(self.conv_channels * scale_factor)

                params.append({
                    'input_channel': in_ch,
                    'output_channel': out_ch,
                    'max_pool': self.pool_size
                })

        else:
            # All layers same channel size
            for i in range(self.num_conv_layers):
                params.append({
                    'input_channel': 1 if i == 0 else self.conv_channels,
                    'output_channel': self.conv_channels,
                    'max_pool': self.pool_size
                })

        return params

    def build_encoder(self, kernel_size, dilation):
        encoder = nn.Sequential()

        for idx, param in enumerate(self.conv_param):
            encoder.add_module(
                f'conv_{idx}',
                Conv2DBlock(
                    in_channels=param['input_channel'],
                    out_channels=param['output_channel'],
                    kernel_size=kernel_size,
                    padding='same',
                    dilation=dilation
                )
            )

            if self.dropout > 0:
                encoder.add_module(f'dropout_{idx}', nn.Dropout2d(self.dropout))

            if self.pool_size is not None:
                encoder.add_module(f'pool_{idx}', nn.MaxPool2d(param['max_pool']))

        return encoder

    def calc_output_freq(self):
        '''Calculate frequency dimension after CNN'''
        if self.pool_size is None:
            return self.n_mels
        else:
            # Each pool layer reduces by pool_size[0]
            return self.n_mels // (self.pool_size[0] ** self.num_conv_layers)

    def calc_output_time(self):
        '''Calculate time dimension after CNN'''
        if self.pool_size is None or self.pool_size[1] == 1:
            return self.n_frames
        else:
            # Each pool layer reduces by pool_size[1]
            return self.n_frames // (self.pool_size[1] ** self.num_conv_layers)

    def forward(self, x):
        if x.ndim == 3:
            x=x.unsqueeze(1)

        x = self.encoder(x) #bcft
        b, _, _, t = x.shape
        x = x.permute(0,3,1,2) #b,t,f,c
        x = x.reshape(b, t, -1)
        x, _ = self.gru(x)
        x = torch.max(x,dim=1)[0]
        x = self.fc(x)
        return x


## Model Info

In [ ]:
if __name__ == "__main__":

    # Create model
    model = SegConv2DGRU()

    # Print model info
    print("\n" + "="*60)
    print("MODEL ARCHITECTURE")
    print("="*60)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}\n")

    # Test forward pass
    batch_size = 8
    x = torch.randn(batch_size, 128, 938)  # [B, n_mels, n_frames]

    print(f"Input shape: {x.shape}")

    model.eval()
    with torch.no_grad():
        output = model(x)

    print(f"Output shape: {output.shape}")
    print(f"Output (logits): {output[0]}")

    # Test with 4D input
    x_4d = torch.randn(batch_size, 1, 128, 938)
    with torch.no_grad():
        output_4d = model(x_4d)

    print(f"\n4D Input shape: {x_4d.shape}")
    print(f"4D Output shape: {output_4d.shape}")

    # Visualize architecture
    print("\n" + "="*60)
    print("LAYER-BY-LAYER OUTPUT SHAPES")
    print("="*60)

    x = torch.randn(1, 1, 128, 938)
    print(f"Input:           {list(x.shape)}")

    # Through encoder
    x = model.encoder(x)
    print(f"After CNN:       {list(x.shape)}")

    # Reshape
    b, c, f, t = x.shape
    x = x.permute(0, 3, 1, 2).reshape(b, t, -1)
    print(f"After reshape:   {list(x.shape)}")

    # Through GRU
    x, _ = model.gru(x)
    print(f"After GRU:       {list(x.shape)}")

    # Max pooling
    x = torch.max(x, dim=1)[0]
    print(f"After max pool:  {list(x.shape)}")

    # FC
    x = model.fc(x)
    print(f"After FC:        {list(x.shape)}")
    print("="*60 + "\n")

# Trainer 

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from abc import ABC, abstractmethod


import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

class MetricsCalculator:
    @staticmethod
    def calculate_binary_metrics(predictions, targets):
        predictions_flat = np.array(predictions).flatten()
        targets_flat = np.array(targets).flatten()

        accuracy = 100 * (predictions_flat == targets_flat).sum() / len(targets_flat)
        precision = precision_score(targets_flat, predictions_flat, zero_division=0)
        recall = recall_score(targets_flat, predictions_flat, zero_division=0)
        f1 = f1_score(targets_flat, predictions_flat, zero_division=0)

        return accuracy, precision, recall, f1

    @staticmethod
    def calculate_per_class_accuracy(predictions, targets):
        predictions_flat = np.array(predictions).flatten()
        targets_flat = np.array(targets).flatten()

        class_0_mask = targets_flat == 0
        class_1_mask = targets_flat == 1

        class_0_acc = 100 * (predictions_flat[class_0_mask] == targets_flat[class_0_mask]).mean() if class_0_mask.sum() > 0 else 0.0
        class_1_acc = 100 * (predictions_flat[class_1_mask] == targets_flat[class_1_mask]).mean() if class_1_mask.sum() > 0 else 0.0

        return class_0_acc, class_1_acc


class BaseTrainer(ABC):
    def __init__(self, model, train_loader, val_loader, device='cuda', lr=0.001):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.lr = lr

        self.model.to(device)
        self.optimizer = Adam(self.model.parameters(), lr=lr)
        self.criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([0.5], device=device)
        )
        self.metrics_calculator = MetricsCalculator()

    def train_epoch(self, epoch):
        """Train mode model for one epoch"""
        self.model.train()

        train_loss = 0.0
        all_predictions = []
        all_targets = []

        print(f"\n{'='*50}")
        print(f"[MODE PRETRAIN EPOCH {epoch+1}] Training Mode Model")
        print(f"{'='*50}")

        for batch in tqdm(self.train_loader, desc=f"Epoch {epoch+1}"):
            mel_spec, mode_labels, _ = batch  #
            mel_spec = mel_spec.to(self.device).float()
            mode_labels = mode_labels.to(self.device).float()

            # Forward pass with ground truth note labels
            self.optimizer.zero_grad()
            mode_output = self.model(mel_spec)
            loss = self.criterion(mel_spec, mode_labels)
            loss.backward()
            self.optimizer.step()

            train_loss += loss.item()

            # Collect predictions
            predictions = (mode_output > 0.5).int()
            all_predictions.extend(predictions.cpu().numpy().flatten().tolist())
            all_targets.extend(mode_labels.cpu().numpy().flatten().tolist())

        # Calculate metrics
        avg_loss = train_loss / len(self.train_loader)
        accuracy, precision, recall, f1 = self.metrics_calculator.calculate_binary_metrics(
            all_predictions, all_targets
        )

        class_0_acc, class_1_acc = self.metrics_calculator.calculate_per_class_accuracy(
            all_predictions, all_targets
        )

        return {
            'loss': avg_loss,
            'accuracy': accuracy,
            'class_0_acc': class_0_acc,
            'class_1_acc': class_1_acc,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }

    def validate_epoch(self, epoch):
        """Validate mode model for one epoch"""
        self.model.eval()

        val_loss = 0.0
        all_predictions = []
        all_targets = []

        print(f"\n[Mode Pretrain Validation] Epoch {epoch+1}")

        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc=f"Validating Epoch {epoch+1}"):
                mel_spec, mode_labels, _ = batch
                mel_spec = mel_spec.to(self.device).float()

                # Forward pass with ground truth note labels
                mode_output = self.model(mel_spec)
                mode_output = mode_output.squeeze(-1) if mode_output.dim() > 1 else mode_output

                loss = self.criterion(mode_output, mode_labels)
                val_loss += loss.item()

                # Collect predictions
                predictions = (mode_output > 0.5).float()
                all_predictions.extend(predictions.cpu().numpy().tolist())
                all_targets.extend(mode_labels.cpu().numpy().tolist())

        # Calculate metrics
        avg_loss = val_loss / len(self.val_loader)
        accuracy, precision, recall, f1 = self.metrics_calculator.calculate_binary_metrics(
            all_predictions, all_targets
        )
        class_0_acc, class_1_acc = self.metrics_calculator.calculate_per_class_accuracy(
            all_predictions, all_targets
        )

        return {
            'loss': avg_loss,
            'accuracy': accuracy,
            'class_0_acc': class_0_acc,
            'class_1_acc': class_1_acc,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }

    def train(self, num_epochs):
        for epoch in range(num_epochs):
            train_metrics = self.train_epoch(epoch)
            val_metrics = self.validate_epoch(epoch)


# Train 

In [ ]:
import os
import torch
from datetime import datetime
import hydra
import wandb
import random
from omegaconf import OmegaConf
from torch.utils.data import DataLoader

T = datetime.now().strftime('%m%d_%H%M%S')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def setup_datasets():
    """Setup train and validation datasets"""
    audio_dir = cfg.data.dir.full_f0_dir

    song_list = [i for i in os.listdir(audio_dir)]

    # Split songs into train and val sets
    train_split_ratio = 0.8
    train_size = int(train_split_ratio * len(song_list))

    random.seed(42)
    song_list_shuffled = song_list.copy()
    random.shuffle(song_list_shuffled)

    train_songs = song_list_shuffled[:train_size]
    val_songs = song_list_shuffled[train_size:]

    train_dataset = BaseDataset(
        data_dir=audio_dir,
        song_list=train_songs
        is_train=True
    )

    val_dataset = Basedataset(
        data_dir=audio_dir,
        song_list=val_songs
        is_train=False
    )

    print(f"Full dataset - Train: {len(full_train_dataset)}, Val: {len(full_val_dataset)}")
    print(f"Mode dataset - Train: {len(mode_train_dataset)}, Val: {len(mode_val_dataset)}")

    return full_train_dataset, full_val_dataset, mode_train_dataset, mode_val_dataset

def setup_dataloaders(full_train_dataset, full_val_dataset,
                      mode_train_dataset, mode_val_dataset, batch_size):
    """Create dataloaders from datasets"""
    full_train_loader = DataLoader(
        full_train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        collate_fn=collate_fn_skip_none
    )

    full_val_loader = DataLoader(
        full_val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn_skip_none
    )

    mode_train_loader = DataLoader(
        mode_train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    mode_val_loader = DataLoader(
        mode_val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    print(f"Full Train batches: {len(full_train_loader)}, Val batches: {len(full_val_loader)}")
    print(f"Mode Train batches: {len(mode_train_loader)}, Val batches: {len(mode_val_loader)}")

    return full_train_loader, full_val_loader, mode_train_loader, mode_val_loader


def setup_models(cfg):
    """Initialize note and mode models"""
    note_model = NoteClassifier(
        conv_channels=[128, 64, 32, 32],
        gru_hidden_size=64,
        gru_num_layers=1,
        dropout_rate=0.5
    )

    mode_model = ModeClassifier(
        input_dim=28,
        d_model=64,
        nhead=4,
        num_layers=2
    )

    return note_model, mode_model


def pretrain_note_model(note_model, full_train_loader, full_val_loader, cfg):
    """Pretrain the note classification model"""
    print("\n" + "="*70)
    print("NOTE MODEL PRETRAINING")
    print("="*70)

    checkpoint_dir = cfg.train.get('checkpoint_dir', './checkpoints')

    note_pretrainer = NotePretrainer(
        note_model=note_model,
        train_loader=full_train_loader,
        val_loader=full_val_loader,
        device=device,
        lr=0.001,
        checkpoint_dir=checkpoint_dir
    )

    note_pretrainer.train(num_epochs=100)

    print(f"Note pretraining completed!")
    print(f"Best F1: {note_pretrainer.checkpoint_manager.best_val_f1:.4f} "
          f"at epoch {note_pretrainer.checkpoint_manager.best_epoch + 1}")

    return note_pretrainer.optimizer


def pretrain_mode_model(mode_model, full_train_loader, full_val_loader, cfg):
    """Pretrain the mode classification model"""
    print("\n" + "="*70)
    print("MODE MODEL PRETRAINING")
    print("="*70)

    checkpoint_dir = cfg.train.get('checkpoint_dir', './checkpoints')

    mode_pretrainer = ModePretrainer(
        mode_model=mode_model,
        train_loader=full_train_loader,
        val_loader=full_val_loader,
        device=device,
        lr=cfg.train.lr,
        checkpoint_dir=checkpoint_dir
    )

    mode_pretrainer.train(num_epochs=50)

    print(f"Mode pretraining completed!")
    print(f"Best F1: {mode_pretrainer.checkpoint_manager.best_val_f1:.4f} "
          f"at epoch {mode_pretrainer.checkpoint_manager.best_epoch + 1}")

    return mode_pretrainer.optimizer


def joint_train(note_model, mode_model,
                full_train_loader, full_val_loader,
                mode_train_loader, mode_val_loader,
                note_optimizer, mode_optimizer, cfg):
    """Joint training of note and mode models"""
    print("\n" + "="*70)
    print("JOINT TRAINING")
    print("="*70)

    checkpoint_dir = cfg.train.get('checkpoint_dir', './checkpoints')
    note_checkpoint_path = cfg.train.get('note_checkpoint_path', None)
    mode_checkpoint_path = cfg.train.get('mode_checkpoint_path', None)
    freeze_note_model = cfg.train.get('freeze_note_model', False)

    trainer = JointTrainer(
        note_model=note_model,
        mode_model=mode_model,
        full_train_loader=full_train_loader,
        full_val_loader=full_val_loader,
        mode_train_loader=mode_train_loader,
        mode_val_loader=mode_val_loader,
        device=device,
        lr=cfg.train.lr,
        pretrained_note_optimizer=note_optimizer,
        pretrained_mode_optimizer=mode_optimizer,
        note_checkpoint_path=note_checkpoint_path,
        mode_checkpoint_path=mode_checkpoint_path,
        freeze_note_model=freeze_note_model,
        checkpoint_dir=checkpoint_dir
    )

    trainer.train(num_epochs=cfg.train.epoch)

    print(f"Joint training completed!")


@hydra.main(config_path="./configs", config_name="config")
def main(cfg):
    run_name = f"PansoriMode_{T}"
    wandb.init(
        project='Pansori_MoCLa',
        name=run_name,
        dir=os.getcwd(),
        mode=cfg.train.get('wandb_mode', 'online')
    )
    wandb.config.update(OmegaConf.to_container(cfg, resolve=True))

    # Setup datasets and dataloaders
    full_train_dataset, full_val_dataset, mode_train_dataset, mode_val_dataset = setup_datasets(cfg)
    full_train_loader, full_val_loader, mode_train_loader, mode_val_loader = setup_dataloaders(
        full_train_dataset, full_val_dataset,
        mode_train_dataset, mode_val_dataset,
        cfg.train.batch_size
    )

    # Initialize models
    note_model, mode_model = setup_models(cfg)

    # Pretrain note model
    if cfg.train.get('do_note_pretrain', True):
        note_optimizer = pretrain_note_model(
            note_model, full_train_loader, full_val_loader, cfg
        )
    else:
        note_optimizer = None
        print("Skipping note model pretraining")

    # Pretrain mode model
    if cfg.train.get('do_mode_pretrain', True):
        mode_optimizer = pretrain_mode_model(
            mode_model, full_train_loader, full_val_loader, cfg
        )
    else:
        mode_optimizer = None
        print("Skipping mode model pretraining")

    # Joint training
    if cfg.train.get('do_joint_train', True):
        joint_train(
            note_model, mode_model,
            full_train_loader, full_val_loader,
            mode_train_loader, mode_val_loader,
            note_optimizer, mode_optimizer, cfg
        )
    else:
        print("Skipping joint training")

    print("\n" + "="*70)
    print("TRAINING PIPELINE COMPLETE!")
    print("="*70)

    if wandb.run is not None:
        wandb.finish()


if __name__ == "__main__":
    main()